<img src="https://raw.githubusercontent.com/pasdptt/public_teaching_tools4DS/main/assets/brand/logo-160.png" width="80" align="right" alt="Course logo">

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pasdptt/public_teaching_tools4DS/blob/main/modules/m06-nonlinear-representations/lab/lab6_nonlinear_representations.ipynb)

# Lab 6 — Nonlinear and learned representations

In [ ]:
# Setup: run this cell first (Google Colab or a local Jupyter kernel).
COURSE_REF = "main"  # version of the course repository to install the helper package from
try:
    import tsds
except ImportError:  # first run in a fresh Colab runtime: install the course helper package
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    f"tsds @ git+https://github.com/pasdptt/public_teaching_tools4DS@{COURSE_REF}#subdirectory=src"], check=True)
    import tsds

tsds.env.check()          # prints versions; warns if a package is outside the tested range
SEED = tsds.seed_everything()
rng = tsds.rng()

import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE, Isomap, trustworthiness
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.exceptions import ConvergenceWarning

**Course:** 040563107 Tools for Statistical Data Science · **Module m06** · **Time:** 3 hours · **Practice lab** (not graded; use the self-checks) · includes an **AI-audit task**

**After this lab you can:**
* compare PCA, Isomap and t-SNE views of the same data with a neighbourhood-preservation measure, trustworthiness (objective m06-O1, O2);
* separate **visualisation quality** from **predictive usefulness** with held-out k-NN accuracy (O2);
* audit an AI-generated pipeline that uses t-SNE coordinates as features, and show why they are not features (O1, AI audit);
* train a small CPU autoencoder with a correct output layer and loss, and compare its reconstruction error with PCA (O3);
* show why a softmax output layer cannot reconstruct pixel intensities (O3, failure analysis).

**Two steps:** in the lab session we **run through** this lab together. Then, as homework, you **do it again on your own** from a fresh
copy and upload the `.ipynb` to the Lab 6 practice assignment in Microsoft Teams.

**How to work:**
1. Run the setup cell first.
2. Complete every cell marked **Task**. Replace `raise NotImplementedError()` with your code, then run the **Self-check** cell below it; it prints `OK` when your answer passes.
3. Write the answers marked **Q** after *Your answer:* (at most 100 words each).
4. Before you finish, use *Runtime → Restart session and run all*. The notebook must run top to bottom without errors (about 2 minutes on a CPU).
5. Fill in the **AI-use log** at the end.

## Part 1 — Three views of the digits (about 35 minutes)

The digits data hold 1,797 images of 8 × 8 pixels (64 variables, intensities 0–16; we rescale them to 0–1). We look at three
two-dimensional **views**:

* **PCA**: the best linear projection for variance;
* **Isomap**: preserves geodesic distances along a neighbourhood graph;
* **t-SNE**: preserves *who is near whom* (local neighbourhoods); distances between clusters and cluster sizes mean little.

**Trustworthiness** (between 0 and 1) asks: are the points that are close in the 2-D view also close in the original 64
dimensions? It penalises "false neighbours" that the view brings together. `sklearn.manifold.trustworthiness(X, Z, n_neighbors=10)`.

In [ ]:
Xd, yd = tsds.data.load("digits")
X = Xd.to_numpy(dtype=float) / 16.0
y = yd.to_numpy()
print(X.shape, "classes:", np.unique(y))


def show_embeddings(emb, y):
    fig, axes = plt.subplots(1, len(emb), figsize=(5 * len(emb), 4.5))
    for ax, (name, Z) in zip(np.atleast_1d(axes), emb.items()):
        sc = ax.scatter(Z[:, 0], Z[:, 1], c=y, cmap="tab10", s=6)
        ax.set_title(name)
        ax.set_xticks([])
        ax.set_yticks([])
    fig.colorbar(sc, ax=axes, label="digit")
    plt.show()

In [ ]:
# Task T1
# (a) emb: dict name -> 2-D coordinates of ALL 1,797 images (array of shape (1797, 2)), for
#       "pca":    PCA(2, random_state=SEED)
#       "isomap": Isomap(n_neighbors=10, n_components=2)
#       "tsne":   TSNE(2, perplexity=30, init="pca", random_state=SEED)
# (b) trust: dict name -> trustworthiness(X, emb[name], n_neighbors=10)
# (c) show_embeddings(emb, y)
# YOUR CODE HERE
raise NotImplementedError()
print({name: round(float(t), 3) for name, t in trust.items()})

In [ ]:
# Self-check for task T1: run it; it should print OK.
assert set(emb) == {"pca", "isomap", "tsne"} and all(Z.shape == (1797, 2) for Z in emb.values())
assert set(trust) == set(emb)
assert abs(trust["pca"] - 0.830) < 0.01, "PCA(2) on X / 16: trustworthiness about 0.83"
assert trust["tsne"] > 0.98, "t-SNE is built to preserve neighbourhoods"
assert trust["tsne"] > trust["isomap"] > trust["pca"] - 0.01
print("OK")

## Part 2 — A good picture is not a good feature set (about 30 minutes)

A view that preserves neighbourhoods well is not automatically useful for **prediction**, and a feature extractor must be able
to transform **new** observations. We estimate held-out accuracy of a 5-nearest-neighbour classifier with every fitted step
inside the pipeline.

In [ ]:
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
knn = KNeighborsClassifier(n_neighbors=5)
print("Can t-SNE transform new points?", hasattr(TSNE(), "transform"))
print("Can Isomap transform new points?", hasattr(Isomap(), "transform"))

In [ ]:
# Task T2
# knn_acc: dict name -> mean 5-fold CV accuracy on (X, y) with cv=cv5 for
#   "majority": DummyClassifier(strategy="most_frequent")
#   "raw64":    knn on all 64 pixels
#   "pca2":     make_pipeline(PCA(2, random_state=SEED), knn)
#   "pca10":    make_pipeline(PCA(10, random_state=SEED), knn)
#   "isomap2":  make_pipeline(Isomap(n_neighbors=10, n_components=2), knn)
# YOUR CODE HERE
raise NotImplementedError()
pd.Series(knn_acc).round(3)

In [ ]:
# Self-check for task T2: run it; it should print OK.
assert set(knn_acc) == {"majority", "raw64", "pca2", "pca10", "isomap2"}
assert abs(knn_acc["majority"] - 0.101) < 0.01
assert abs(knn_acc["pca2"] - 0.620) < 0.02, "PCA(2) inside the pipeline: about 0.62"
assert knn_acc["pca10"] > 0.95 and knn_acc["raw64"] > 0.95
assert knn_acc["pca2"] < knn_acc["isomap2"] < knn_acc["pca10"]
print("OK")

**Q1.** **Interpretation.** Use your trustworthiness values (Part 1) and accuracies (Part 2). Which view gives the best picture, and which
representation gives the best held-out prediction? Why can t-SNE not appear in the Part 2 table at all?

*Your answer:* 

## Part 3 — AI audit: t-SNE coordinates as features (about 30 minutes)

A student asked an AI assistant for *"a high-accuracy digits classifier using t-SNE"*. The code below is representative of what
such tools produce (it is the same mistake as in the course's old feature-extraction notebook, audit issue M11). Run it.

In [ ]:
# ---- AI-generated code (do not edit) ----
tsne_all = TSNE(n_components=2, random_state=SEED).fit_transform(X)    # embed all digits
rf = RandomForestClassifier(n_estimators=200, random_state=SEED)
ai_score = cross_val_score(rf, tsne_all, y, cv=cv5).mean()
print(f"Accuracy with t-SNE features: {ai_score:.3f}  -> t-SNE features are excellent!")
# ---- end of AI-generated code ----

The accuracy is real for **these** coordinates. The question is whether the coordinates are a feature extractor. To use the
model on a new digit you would have to run t-SNE again with the new digit included, and t-SNE's output depends on its random
start. We test exactly that: two t-SNE runs on the same images that differ only in the random start.

In [ ]:
idx_tr, idx_te = train_test_split(np.arange(len(y)), test_size=0.25, stratify=y, random_state=SEED)

In [ ]:
# Task T3
# (a) run_a, run_b: TSNE(2, perplexity=30, init="random", random_state=r).fit_transform(X), with r = SEED and r = SEED + 1
# (b) fit RandomForestClassifier(n_estimators=200, random_state=SEED) on run_a[idx_tr], y[idx_tr]
# (c) acc_same:  its accuracy on run_a[idx_te], y[idx_te]
#     acc_cross: its accuracy on run_b[idx_te], y[idx_te] (the same test images, embedded by a second run)
# YOUR CODE HERE
raise NotImplementedError()
print(f"same run {acc_same:.3f} | second run of t-SNE {acc_cross:.3f}")

In [ ]:
# Self-check for task T3: run it; it should print OK.
assert run_a.shape == run_b.shape == (1797, 2)
assert acc_same > 0.95, "within one run the clusters are well separated"
assert acc_cross < 0.6, "the model should not transfer to a second t-SNE run"
print("OK")

**Q2.** **AI audit.** List at least three problems with the AI-generated code, using your numbers. Then say what you would do instead
(a) to **show** the digits in 2-D and (b) to **classify** new digits.

*Your answer:* 

## Part 4 — An autoencoder as nonlinear PCA (about 35 minutes)

An **autoencoder** is a neural network trained to reproduce its input through a narrow middle layer (the code). With pixel
intensities in $[0, 1]$, a correct set-up is an output layer with **identity** (linear) activation and **squared error** loss; this is
what scikit-learn's `MLPRegressor` does, on a CPU. We compare reconstruction error on **held-out** images with PCA using the same
code size $k = 10$:

$$\text{MSE} = \frac{1}{n_{\text{test}}\,p}\sum_i \lVert x_i - \hat x_i \rVert^2 .$$

Theory: a **linear** autoencoder with squared error can do no better than PCA with the same $k$ (it learns the same subspace).
A **nonlinear** one can do better, if it is trained long enough.

In [ ]:
X_tr, X_te = X[idx_tr], X[idx_te]
K = 10


def mse(A, B):
    return float(np.mean((A - B) ** 2))

In [ ]:
# Task T4
# rec_mse: dict name -> test MSE (use mse) after fitting on X_tr only, for
#   "pca":       PCA(K, random_state=SEED); reconstruct with inverse_transform(transform(X_te))
#   "linear_ae": MLPRegressor(hidden_layer_sizes=(K,), activation="identity", max_iter=3000, tol=1e-7,
#                             n_iter_no_change=50, random_state=SEED), fitted with X_tr as input AND target
#   "tanh_ae":   as linear_ae but hidden_layer_sizes=(64, K, 64), activation="tanh"
# Time the tanh autoencoder: it should take well under a minute on a CPU.
warnings.filterwarnings("ignore", category=ConvergenceWarning)
# YOUR CODE HERE
raise NotImplementedError()
pd.Series(rec_mse).round(4)

In [ ]:
# Self-check for task T4: run it; it should print OK.
assert set(rec_mse) == {"pca", "linear_ae", "tanh_ae"}
assert abs(rec_mse["pca"] - 0.0196) < 0.001, "PCA(10) test MSE on X / 16: about 0.0196"
assert abs(rec_mse["linear_ae"] - rec_mse["pca"]) < 0.001, "a well-trained linear autoencoder matches PCA"
assert rec_mse["tanh_ae"] < rec_mse["pca"], "the nonlinear autoencoder should beat PCA here"
print("OK")

## Part 5 — Failure analysis: a softmax output layer (about 25 minutes)

The course's old notebook built its autoencoder with a **softmax** output and binary cross-entropy (audit issue M9). A softmax
layer returns non-negative numbers that **sum to 1** in every row. Pixel rows here sum to much more than 1.

For any output $\hat x$ with $\sum_j \hat x_j = 1$, the Cauchy–Schwarz inequality gives
$\lVert x - \hat x\rVert^2 \ge \left(\sum_j x_j - 1\right)^2 / p$. So whatever the network learns, its MSE on a test set is at least

$$\text{bound} = \frac{1}{n_{\text{test}}}\sum_i \frac{\left(\sum_j x_{ij} - 1\right)^2}{p^2}.$$

In [ ]:
# Task T5
# (a) row_sums: the sum of each row of X_te (shape (n_test,))
# (b) softmax_bound: the bound above, a float (p = 64)
# YOUR CODE HERE
raise NotImplementedError()
print(f"mean row sum {row_sums.mean():.2f} | any softmax output has test MSE of at least {softmax_bound:.4f}")

In [ ]:
# Self-check for task T5: run it; it should print OK.
assert row_sums.shape == (X_te.shape[0],)
assert abs(softmax_bound - 0.0848) < 0.002, "check the formula: squared (row sum - 1), divided by p squared, averaged"
print("OK")

**Q3.** **Failure analysis.** Explain, with your numbers from Parts 4 and 5, why the softmax autoencoder was bound to fail and why the
course's old notebook did not notice. What output layer and loss would you use for (a) intensities in $[0, 1]$ and (b) standardised
inputs that can be negative?

*Your answer:* 

## AI-use log (required; see the course AI policy)

| Tool | What you asked or used it for | What you accepted or changed | How you verified it |
|---|---|---|---|
| *e.g. Colab AI* | *how to call trustworthiness in scikit-learn* | *used n_neighbors=10 as the lab asks* | *T1 self-check* |
| | | | |

If you used no AI tool, write "None".

**Cite the data** when you use them: Alpaydin, E. & Kaynak, C. (1998). Optical Recognition of Handwritten Digits [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C50P49 (CC BY 4.0).